# 04 — Retrieval
Menggunakan LangChain Retriever interface untuk query ke Chroma.
Interface standar yang akan dipakai di MCP server dan chatbot nanti.

## 0. Load Chroma + Embedding Model
Load vector store dan model dari disk — tidak perlu embed ulang.

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-large"
)

# Load Chroma dari disk
vectorstore = Chroma(
    collection_name="phis_sds",
    embedding_function=embeddings,
    persist_directory="../vector_db"
)

print(f"Vector store loaded!")
print(f"Total vectors : {vectorstore._collection.count()}")

C:\Users\Dev\AppData\Local\Temp\ipykernel_17352\3850646739.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\Dev\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2279.72it/s]
C:\Users\Dev\AppData\Local\Temp\ipykernel_17352\3850646739.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the cl

Vector store loaded!
Total vectors : 666


## 1. Buat Retriever
Convert vector store ke LangChain Retriever interface.

In [2]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"Retriever siap!")
print(f"Search type : similarity")
print(f"Top-K       : 3")

Retriever siap!
Search type : similarity
Top-K       : 3


## 2. Query via Retriever
Test retriever dengan natural language query.

In [3]:
def retrieve(query):
    docs = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"{'='*55}")
    for i, doc in enumerate(docs):
        print(f"Rank {i+1} | Halaman: {doc.metadata['page']}")
        print(f"{doc.page_content[:200]}")
        print(f"{'-'*55}")

retrieve("what is special approval medicine?")

Query: 'what is special approval medicine?'
Rank 1 | Halaman: 0
SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
DOCUMENT ID : PhIS/SDS/SAM
-------------------------------------------------------
Rank 2 | Halaman: 3
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page iii
DOCUMENT VERIFICATION AND AUTHORIZATION
System Design Specification (SDS) for
Special Approval Medicine (SAM)
The enclosed document has bee
-------------------------------------------------------
Rank 3 | Halaman: 129
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 118
a. On click of ‘Approve’
 Record status is updated to ‘Approved by Director
General of Health (DGH)/Senior Director of
Pharmaceutical Serv
-------------------------------------------------------


## 3. MMR Search — Hasil Lebih Beragam
Maximal Marginal Relevance — hindari hasil yang terlalu mirip satu sama lain.

In [4]:
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

def retrieve_mmr(query):
    docs = retriever_mmr.invoke(query)
    print(f"Query (MMR): '{query}'")
    print(f"{'='*55}")
    for i, doc in enumerate(docs):
        print(f"Rank {i+1} | Halaman: {doc.metadata['page']}")
        print(f"{doc.page_content[:200]}")
        print(f"{'-'*55}")

retrieve_mmr("what is special approval medicine?")

Query (MMR): 'what is special approval medicine?'
Rank 1 | Halaman: 0
SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
DOCUMENT ID : PhIS/SDS/SAM
-------------------------------------------------------
Rank 2 | Halaman: 84
actions or modifications will not be permissible.
 Rejected notification will be displayed on the Dashboard for
requester.
c. On click on ‘Review’
 This is an optional process depending on the user’
-------------------------------------------------------
Rank 3 | Halaman: 81
Secretariat for evaluation.
 Non-MOH Specialist – Hospital role
o Record Status updated to ‘Pending Review by Pharmacist’.
o Request shall flow to Non – MOH Pharmacist for review
 Non-MOH Medical Pr
-------------------------------------------------------


## 4. Filter by Metadata
Cari dokumen dari halaman tertentu saja.
Useful untuk query yang lebih spesifik.

In [5]:
def retrieve_filtered(query, page_number):
    docs = vectorstore.similarity_search(
        query,
        k=3,
        filter={"page": page_number}
    )
    print(f"Query  : '{query}'")
    print(f"Filter : halaman {page_number}")
    print(f"{'='*55}")
    for i, doc in enumerate(docs):
        print(f"Rank {i+1} | Halaman: {doc.metadata['page']}")
        print(f"{doc.page_content[:200]}")
        print(f"{'-'*55}")

# Cari di halaman 41 saja
retrieve_filtered("SAM request list", page_number=41)

Query  : 'SAM request list'
Filter : halaman 41
Rank 1 | Halaman: 41
that are pending for action for respective role.
 Approved – Shall display list of approved SAM Request that involves the
respective user.
 Rejected – Shall display list of rejected SAM Request that
-------------------------------------------------------
Rank 2 | Halaman: 41
System Design Specification (SDS)
Ref: PhIS/SDS/SAM Page 30
Request
 Navigate to ‘Overall Request Summary’ tab to view overall list of SAM Request
by whole facility or by same role.
 Navigate to ‘Bu
-------------------------------------------------------
Rank 3 | Halaman: 41
Not
Recommended/
Not Endorsed/
Rejected by
Facility/ HQ level
 Submitted
SAM
Request
that is
being
processed
by Facility/
HQ
HOD  Draft
 Return for
Revision by
Pharmacist
Pharmacist  Draft
 Retur
-------------------------------------------------------


## 5. Summary Retrieval Methods
Ringkasan tiga cara query yang sudah kita pelajari.

In [6]:
print("="*55)
print("SUMMARY RETRIEVAL METHODS")
print("="*55)
print("""
1. SIMILARITY SEARCH
   → Ambil k dokumen paling mirip query
   → Cocok untuk: cari info spesifik
   → retriever = vectorstore.as_retriever(
         search_type="similarity", k=3)

2. MMR SEARCH  
   → Ambil dokumen relevan tapi beragam
   → Cocok untuk: konteks yang lebih luas
   → retriever = vectorstore.as_retriever(
         search_type="mmr", k=3, fetch_k=10)

3. FILTER METADATA
   → Cari di halaman/section tertentu saja
   → Cocok untuk: query yang sangat spesifik
   → vectorstore.similarity_search(
         query, filter={"page": 41})
""")
print("="*55)
print("Next: 05_mcp_server.ipynb")
print("="*55)

SUMMARY RETRIEVAL METHODS

1. SIMILARITY SEARCH
   → Ambil k dokumen paling mirip query
   → Cocok untuk: cari info spesifik
   → retriever = vectorstore.as_retriever(
         search_type="similarity", k=3)

2. MMR SEARCH  
   → Ambil dokumen relevan tapi beragam
   → Cocok untuk: konteks yang lebih luas
   → retriever = vectorstore.as_retriever(
         search_type="mmr", k=3, fetch_k=10)

3. FILTER METADATA
   → Cari di halaman/section tertentu saja
   → Cocok untuk: query yang sangat spesifik
   → vectorstore.similarity_search(
         query, filter={"page": 41})

Next: 05_mcp_server.ipynb
